In [6]:
# %% CELL 9c: RANKGAUSS PREPROCESSING (The NN Savior)
# This forces features into a Normal Distribution, which NNs love.
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer

print("⚗️  Applying RankGauss Transformation (QuantileTransformer)...")

# 1. Load the Tree Data (Full 221 Features)
DATA_DIR = '../data/processed/trees'
try:
    X_train = np.load(os.path.join(DATA_DIR, 'X_train_tree.npy'))
    X_val   = np.load(os.path.join(DATA_DIR, 'X_val_tree.npy'))
    X_test  = np.load(os.path.join(DATA_DIR, 'X_test_tree.npy'))
    print(f"   ✅ Data Loaded. Shape: {X_train.shape}")
except FileNotFoundError:
    raise ValueError("⚠️ Run Cell 9a first!")

# 2. FIT QUANTILE TRANSFORMER
# output_distribution='normal' is the magic key.
print("   🌊 Transforming distributions to Gaussian (Normal)...")
qt = QuantileTransformer(n_quantiles=2000, output_distribution='normal', random_state=42, subsample=100000)

# Fit on Train, Transform All
X_train_gauss = qt.fit_transform(X_train)
X_val_gauss   = qt.transform(X_val)
X_test_gauss  = qt.transform(X_test)

# 3. SAVE TO SPECIAL NN FOLDER
OUT_DIR = '../data/processed/nn_rankgauss'
os.makedirs(OUT_DIR, exist_ok=True)

np.save(os.path.join(OUT_DIR, 'X_train_nn.npy'), X_train_gauss)
np.save(os.path.join(OUT_DIR, 'X_val_nn.npy'),   X_val_gauss)
np.save(os.path.join(OUT_DIR, 'X_test_nn.npy'),  X_test_gauss)

print(f"   💾 Saved RankGauss data to {OUT_DIR}")
print("   👉 Now update Cell 9.5 to point to this folder!")

⚗️  Applying RankGauss Transformation (QuantileTransformer)...
   ✅ Data Loaded. Shape: (125452, 221)
   🌊 Transforming distributions to Gaussian (Normal)...
   💾 Saved RankGauss data to ../data/processed/nn_rankgauss
   👉 Now update Cell 9.5 to point to this folder!


In [9]:
# %% CELL 9.5: FINAL NN (RankGauss Inputs + Scaled Targets)
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import copy
import os

print("💎 Launching FINAL NN (RankGauss Inputs + Scaled Targets)...")

# 1. SETUP
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.backends.mps.is_available(): DEVICE = torch.device("mps")

# 2. LOAD RANKGAUSS DATA (From Cell 9c)
# This is the critical change: We use the Gaussian-distributed features
NN_DIR = '../data/processed/nn_rankgauss'
SHARED_DIR = '../data/processed'

try:
    X_train = np.load(os.path.join(NN_DIR, 'X_train_nn.npy'))
    X_val   = np.load(os.path.join(NN_DIR, 'X_val_nn.npy'))
    X_test  = np.load(os.path.join(NN_DIR, 'X_test_nn.npy'))
    
    y_tr    = np.load(os.path.join(SHARED_DIR, 'y_tr.npy'))
    y_val   = np.load(os.path.join(SHARED_DIR, 'y_val.npy'))
    test_ids = np.load(os.path.join(SHARED_DIR, 'test_ids.npy'), allow_pickle=True)
    
    print(f"   ✅ RankGauss Data Loaded. Shape: {X_train.shape}")
except FileNotFoundError:
    raise ValueError("⚠️ Data not found. Please run Cell 9c (RankGauss) first!")

# 3. TARGET SCALING (Standardization)
# We bring targets back to N(0,1) so the model learns efficiently
target_mean = y_tr.mean(axis=0)
target_std  = y_tr.std(axis=0)

y_tr_scaled  = (y_tr - target_mean) / (target_std + 1e-8)
y_val_scaled = (y_val - target_mean) / (target_std + 1e-8)

# Convert to Tensors
X_train_t = torch.FloatTensor(np.nan_to_num(X_train, nan=0.0))
y_train_t = torch.FloatTensor(y_tr_scaled)
X_val_t   = torch.FloatTensor(np.nan_to_num(X_val, nan=0.0))
y_val_t   = torch.FloatTensor(y_val_scaled)
X_test_t  = torch.FloatTensor(np.nan_to_num(X_test, nan=0.0))

# 4. ARCHITECTURE: Wide & Shallow (Best for this data)
class WideScaledMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            # Wide Layer to pattern match the Gaussian inputs
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.SiLU(), 
            nn.Dropout(0.2), 
            
            # Bottleneck
            nn.Linear(1024, 256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(0.2),
            
            # Output
            nn.Linear(256, 3)
        )
        
    def forward(self, x): 
        return self.net(x)

# 5. TRAINING LOOP
model = WideScaledMLP(X_train.shape[1]).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.L1Loss() # Optimizing MAE

# Aggressive scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
train_loader = DataLoader(list(zip(X_train_t, y_train_t)), batch_size=2048, shuffle=True)

EPOCHS = 35
best_loss = float('inf')
best_weights = None

print("   🚀 Training Started (Scaled Targets)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t.to(DEVICE))
        val_loss = criterion(val_pred, y_val_t.to(DEVICE)).item()
    
    # Update LR and Save Best
    scheduler.step(val_loss)
    if val_loss < best_loss:
        best_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
    
    if (epoch+1) % 5 == 0:
        print(f"   Epoch {epoch+1}/{EPOCHS} | Val MAE (Scaled): {val_loss:.5f} | Best: {best_loss:.5f}")

# 6. PREDICT & SAVE
print("   🔮 Generating Predictions...")
model.load_state_dict(best_weights)
model.eval()

with torch.no_grad():
    pred_scaled = model(X_test_t.to(DEVICE)).cpu().numpy()

# Inverse Transform (Crucial!)
pred_final = (pred_scaled * target_std) + target_mean

# Save Submission
sub_nn = pd.DataFrame({'id': test_ids})
sub_nn[['target_short', 'target_medium', 'target_long']] = pred_final

# Zero-Center Polish
for c in ['target_short', 'target_medium', 'target_long']:
    sub_nn[c] = sub_nn[c] - sub_nn[c].mean()

sub_nn.to_csv('../submissions/submission_nn_final.csv', index=False)
print("✅ Saved '../submissions/submission_nn_final.csv'")

💎 Launching FINAL NN (RankGauss Inputs + Scaled Targets)...
   ✅ RankGauss Data Loaded. Shape: (125452, 221)
   🚀 Training Started (Scaled Targets)...
   Epoch 5/35 | Val MAE (Scaled): 0.58661 | Best: 0.51862
   Epoch 10/35 | Val MAE (Scaled): 0.71572 | Best: 0.51862
   Epoch 15/35 | Val MAE (Scaled): 0.81156 | Best: 0.51862
   Epoch 20/35 | Val MAE (Scaled): 0.84604 | Best: 0.51862
   Epoch 25/35 | Val MAE (Scaled): 0.86074 | Best: 0.51862
   Epoch 30/35 | Val MAE (Scaled): 0.85962 | Best: 0.51862
   Epoch 35/35 | Val MAE (Scaled): 0.86706 | Best: 0.51862
   🔮 Generating Predictions...
✅ Saved '../submissions/submission_nn_final.csv'
